# Phase 2 Final Hindi Training Notebook
This notebook trains the Hindi language model on Kaggle.


## 1. Configuration


In [ ]:

# ---------------------------------------------------------------------------
# Training Budget and Hyperparameters
# ---------------------------------------------------------------------------
MAX_STEPS = 25_000
BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 3e-4
MIN_LEARNING_RATE = 3e-5
WARMUP_STEPS = 2_000
EVAL_INTERVAL = 1_000
CHECKPOINT_INTERVAL = 5_000
LOG_INTERVAL = 50
WEIGHT_DECAY = 0.1
MAX_GRAD_NORM = 1.0
RANDOM_SEED = 42
CONTEXT_LENGTH = 512


## 2. Imports and Source Loading


In [ ]:

import json
import math
import os
import random
import sys
import shutil
import tempfile
import time
from pathlib import Path
from typing import Optional
from datetime import datetime, timezone

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import sentencepiece as spm
import matplotlib.pyplot as plt


In [ ]:
from dataclasses import dataclass

@dataclass
class LMConfig:
    vocab_size: int = 10000
    d_model: int = 448
    n_layers: int = 8
    n_heads: int = 7
    d_ff: int = 2048
    max_seq_len: int = 512
    dropout: float = 0.1
    bias: bool = True


In [ ]:
import math
import torch
import torch.nn as nn

class InputEmbeddings(nn.Module):
    def __init__(self, config: LMConfig):
        super().__init__()
        self.max_seq_len = config.max_seq_len
        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.position_embedding = nn.Embedding(config.max_seq_len, config.d_model)
        self.dropout = nn.Dropout(config.dropout)
        
    def forward(self, input_ids):
        B, T = input_ids.shape
        if T > self.max_seq_len:
            raise ValueError(f"Sequence length {T} exceeds maximum sequence length {self.max_seq_len}")
            
        device = input_ids.device
        position_ids = torch.arange(0, T, dtype=torch.long, device=device)
        
        token_embeddings = self.token_embedding(input_ids)
        position_embeddings = self.position_embedding(position_ids)
        
        x = token_embeddings + position_embeddings
        x = self.dropout(x)
        return x

class CausalSelfAttention(nn.Module):
    def __init__(self, config: LMConfig):
        super().__init__()
        if config.d_model % config.n_heads != 0:
            raise ValueError(f"d_model ({config.d_model}) must be divisible by n_heads ({config.n_heads})")
            
        self.d_model = config.d_model
        self.n_heads = config.n_heads
        self.head_dim = config.d_model // config.n_heads
        self.max_seq_len = config.max_seq_len
        
        self.q_proj = nn.Linear(config.d_model, config.d_model, bias=config.bias)
        self.k_proj = nn.Linear(config.d_model, config.d_model, bias=config.bias)
        self.v_proj = nn.Linear(config.d_model, config.d_model, bias=config.bias)
        self.out_proj = nn.Linear(config.d_model, config.d_model, bias=config.bias)
        
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        
        # Causal mask register_buffer ensures it moves to correct device
        mask = torch.tril(torch.ones(config.max_seq_len, config.max_seq_len)).view(
            1, 1, config.max_seq_len, config.max_seq_len
        )
        self.register_buffer("bias_mask", mask)
        
    def forward(self, x, return_attention=False):
        B, T, C = x.size() # B, T, d_model
        
        if T > self.max_seq_len:
            raise ValueError(f"Sequence length {T} exceeds maximum sequence length {self.max_seq_len}")
            
        # Linear projections
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        
        # Reshape to [B, T, n_heads, head_dim] -> [B, n_heads, T, head_dim]
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        
        # Scaled dot-product attention
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))
        
        # Causal mask applied BEFORE softmax
        att = att.masked_fill(self.bias_mask[:, :, :T, :T] == 0, float('-inf'))
        
        # Softmax
        att = torch.nn.functional.softmax(att, dim=-1)
        
        # Attention probabilities dropout
        att_dropped = self.attn_dropout(att)
        
        # Multiply with values
        y = att_dropped @ v
        
        # Re-assemble all head outputs side by side
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        
        # Output projection
        y = self.resid_dropout(self.out_proj(y))
        
        if return_attention:
            return y, att
        return y


class FeedForward(nn.Module):
    def __init__(self, config: LMConfig):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.d_model, config.d_ff, bias=config.bias),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.d_ff, config.d_model, bias=config.bias),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, config: LMConfig):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.d_model)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.d_model)
        self.ffn = FeedForward(config)

    def forward(self, x, return_attention=False):
        # Pre-LayerNorm: x = x + Attention(LayerNorm(x))
        if return_attention:
            attn_out, att_weights = self.attn(self.ln_1(x), return_attention=True)
            x = x + attn_out
        else:
            x = x + self.attn(self.ln_1(x))

        # Pre-LayerNorm: x = x + FFN(LayerNorm(x))
        x = x + self.ffn(self.ln_2(x))

        if return_attention:
            return x, att_weights
        return x


class DecoderOnlyLM(nn.Module):
    def __init__(self, config: LMConfig):
        super().__init__()
        self.config = config

        # Input representation
        self.embedding = InputEmbeddings(config)

        # Transformer blocks
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layers)])

        # Final LayerNorm
        self.ln_f = nn.LayerNorm(config.d_model)

        # LM head: weight is tied to the token embedding, but has its own bias
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=config.bias)

        # Weight tying: LM head weight IS the token embedding weight
        self.lm_head.weight = self.embedding.token_embedding.weight

    def forward(self, input_ids, return_attention=False):
        # input_ids: [B, T]
        x = self.embedding(input_ids)

        attentions = [] if return_attention else None

        for block in self.blocks:
            if return_attention:
                x, att = block(x, return_attention=True)
                attentions.append(att)
            else:
                x = block(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)  # [B, T, vocab_size]

        if return_attention:
            return logits, attentions
        return logits


In [ ]:
"""
Phase 2 — Stage 5: Data Preparation Pipeline

Converts JSONL corpora into memory-mapped uint16 .bin files for
efficient language model training.

Supports Hindi through configuration arguments.
All paths are supplied externally for portability across
local / Kaggle / Colab environments.
"""

import argparse
import json
import os
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import sentencepiece as spm


# ---------------------------------------------------------------------------
# Tokenization / conversion
# ---------------------------------------------------------------------------

def stream_jsonl_texts(input_dir: Path, text_field: str = "text"):
    """Yield text strings from all .jsonl files in input_dir (sorted for determinism)."""
    jsonl_files = sorted(input_dir.glob("*.jsonl"))
    if not jsonl_files:
        raise FileNotFoundError(f"No .jsonl files found in {input_dir}")
    for jsonl_path in jsonl_files:
        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                doc = json.loads(line)
                text = doc.get(text_field)
                if text and isinstance(text, str) and text.strip():
                    yield text.strip()
                else:
                    yield None  # signal a skip


def convert_split(
    input_dir: Path,
    output_path: Path,
    tokenizer_path: Path,
    text_field: str = "text",
):
    """
    Stream JSONL documents, tokenize with SentencePiece, and write
    token IDs as a flat uint16 binary file.

    Returns a dict of statistics.
    """
    # Load tokenizer
    sp = spm.SentencePieceProcessor()
    sp.Load(str(tokenizer_path))
    vocab_size = sp.GetPieceSize()

    if vocab_size > 65535:
        raise ValueError(
            f"Vocabulary size {vocab_size} exceeds uint16 max (65535)"
        )

    output_path.parent.mkdir(parents=True, exist_ok=True)

    n_docs = 0
    n_skipped = 0
    n_tokens = 0
    token_min = 65535
    token_max = 0

    t0 = time.time()

    with open(output_path, "wb") as out_f:
        for text in stream_jsonl_texts(input_dir, text_field):
            if text is None:
                n_skipped += 1
                n_docs += 1
                continue

            ids = sp.Encode(text, out_type=int)
            if not ids:
                n_skipped += 1
                n_docs += 1
                continue

            arr = np.array(ids, dtype=np.uint16)
            arr.tofile(out_f)

            n_tokens += len(ids)
            token_min = min(token_min, int(arr.min()))
            token_max = max(token_max, int(arr.max()))
            n_docs += 1

    elapsed = time.time() - t0
    file_bytes = output_path.stat().st_size if output_path.exists() else 0

    stats = {
        "input_dir": input_dir.name,
        "output_path": output_path.name,
        "tokenizer": tokenizer_path.name,
        "vocab_size": vocab_size,
        "dtype": "uint16",
        "documents_processed": n_docs,
        "documents_skipped": n_skipped,
        "token_count": n_tokens,
        "token_min": token_min if n_tokens > 0 else None,
        "token_max": token_max if n_tokens > 0 else None,
        "bytes_written": file_bytes,
        "elapsed_seconds": round(elapsed, 3),
        "tokens_per_second": round(n_tokens / elapsed, 1) if elapsed > 0 else 0,
    }

    return stats


def save_metadata(output_dir: Path, language: str, split_stats: dict,
                  tokenizer_name: str):
    """Save metadata.json alongside the .bin files."""
    meta = {
        "language": language,
        "tokenizer_model": tokenizer_name,
        "vocab_size": 10000,
        "dtype": "uint16",
        "context_length": 512,
        "conversion_timestamp": datetime.now(timezone.utc).isoformat(),
        "splits": split_stats,
    }
    meta_path = output_dir / "metadata.json"
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)
    return meta_path


# ---------------------------------------------------------------------------
# Dataset reader (memory-mapped)
# ---------------------------------------------------------------------------

class TokenDataset:
    """
    Memory-mapped dataset over a flat uint16 .bin file.

    Each sample returns:
        x: tokens[i : i + context_length]       (input)
        y: tokens[i+1 : i + context_length + 1]  (target)
    """

    def __init__(self, bin_path: Path, context_length: int = 512):
        self.bin_path = Path(bin_path)
        if not self.bin_path.exists():
            raise FileNotFoundError(f"Binary file not found: {self.bin_path}")

        self.data = np.memmap(self.bin_path, dtype=np.uint16, mode="r")
        self.context_length = context_length
        self.n_tokens = len(self.data)

        # We need at least context_length + 1 tokens for one sample
        if self.n_tokens < context_length + 1:
            raise ValueError(
                f"Token file has {self.n_tokens} tokens but needs at least "
                f"{context_length + 1} for context_length={context_length}"
            )

    def __len__(self):
        """Number of valid starting positions."""
        return self.n_tokens - self.context_length

    def __getitem__(self, idx):
        """Return (x, y) token arrays at position idx."""
        if idx < 0 or idx >= len(self):
            raise IndexError(
                f"Index {idx} out of range for dataset with {len(self)} positions"
            )
        chunk = self.data[idx : idx + self.context_length + 1].astype(np.int64)
        x = chunk[:-1]
        y = chunk[1:]
        return x, y

    def random_sample(self, rng=None):
        """Return a random (x, y) pair."""
        if rng is None:
            rng = np.random.default_rng()
        idx = rng.integers(0, len(self))
        return self[idx]


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------




In [ ]:
"""
Phase 2 — Stage 6: Training Configuration

Separate from the model architecture (LMConfig) so that hyperparameters
can be tuned without touching the model definition.
"""

from dataclasses import dataclass, field
from pathlib import Path


@dataclass
class TrainConfig:
    # -----------------------------------------------------------------------
    # Identity
    # -----------------------------------------------------------------------
    language: str = "hindi"          # "hindi"
    run_name: str = "run"            # used for log/checkpoint naming

    # -----------------------------------------------------------------------
    # Reproducibility
    # -----------------------------------------------------------------------
    seed: int = 42

    # -----------------------------------------------------------------------
    # Data
    # -----------------------------------------------------------------------
    context_length: int = 512
    num_workers: int = 0             # DataLoader workers (0 = main process)

    # -----------------------------------------------------------------------
    # Batch / accumulation
    # -----------------------------------------------------------------------
    batch_size: int = 16             # micro-batch size per step
    grad_accum_steps: int = 1        # effective_batch = batch_size * grad_accum_steps

    # -----------------------------------------------------------------------
    # Optimiser
    # -----------------------------------------------------------------------
    learning_rate: float = 3e-4
    min_lr: float = 3e-5             # cosine decay floor
    weight_decay: float = 0.1
    beta1: float = 0.9
    beta2: float = 0.95
    eps: float = 1e-8

    # -----------------------------------------------------------------------
    # Schedule
    # -----------------------------------------------------------------------
    max_steps: int = 100_000         # total optimiser steps
    warmup_steps: int = 2_000        # linear warmup duration

    # -----------------------------------------------------------------------
    # Regularisation / gradient
    # -----------------------------------------------------------------------
    grad_clip: float = 1.0          # 0.0 disables clipping

    # -----------------------------------------------------------------------
    # Mixed precision
    # -----------------------------------------------------------------------
    use_amp: bool = True             # disabled automatically on CPU

    # -----------------------------------------------------------------------
    # Checkpointing
    # -----------------------------------------------------------------------
    output_dir: Path = Path("checkpoints")
    checkpoint_interval: int = 1_000   # save every N optimiser steps
    keep_latest_n: int = 3            # how many step-checkpoints to retain

    # -----------------------------------------------------------------------
    # Logging / evaluation
    # -----------------------------------------------------------------------
    log_interval: int = 10           # print/log training metrics every N steps
    eval_interval: int = 500         # run validation every N steps
    eval_batches: int = 50           # number of batches for each validation pass

    def __post_init__(self):
        self.output_dir = Path(self.output_dir)

        if self.language not in ("hindi",):
            raise ValueError(f"language must be 'hindi', got {self.language!r}")
        if self.batch_size < 1:
            raise ValueError("batch_size must be >= 1")
        if self.grad_accum_steps < 1:
            raise ValueError("grad_accum_steps must be >= 1")
        if self.max_steps < 1:
            raise ValueError("max_steps must be >= 1")
        if self.warmup_steps < 0:
            raise ValueError("warmup_steps must be >= 0")
        if self.warmup_steps >= self.max_steps:
            raise ValueError("warmup_steps must be < max_steps")
        if not (0.0 < self.learning_rate):
            raise ValueError("learning_rate must be positive")
        if not (0.0 <= self.min_lr < self.learning_rate):
            raise ValueError("min_lr must satisfy 0 <= min_lr < learning_rate")
        if not (0.0 <= self.grad_clip):
            raise ValueError("grad_clip must be >= 0")


In [ ]:
"""
Phase 2 — Stage 6: Training Infrastructure

Implements:
  - seed_everything
  - build_optimizer
  - build_scheduler  (linear warmup + cosine decay)
  - compute_loss / perplexity
  - training_step
  - evaluate
  - save_checkpoint / load_checkpoint
  - JsonlLogger
  - train  (main training loop)
"""

import json
import math
import os
import random
import shutil
import tempfile
import time
from pathlib import Path
from typing import Optional

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset



# ---------------------------------------------------------------------------
# 1. Reproducibility
# ---------------------------------------------------------------------------

def seed_everything(seed: int) -> None:
    """Seed Python, NumPy, PyTorch CPU, and CUDA (if available)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ---------------------------------------------------------------------------
# 2. Optimizer
# ---------------------------------------------------------------------------

def build_optimizer(model: nn.Module, cfg: TrainConfig) -> torch.optim.AdamW:
    """
    Build AdamW with separate weight-decay groups.

    Parameters without meaningful weight decay (biases, LayerNorm weights)
    are placed in a no-decay group.
    """
    decay_params = []
    no_decay_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        # Exclude 1-D parameters (biases, LN weights/biases) from WD
        if param.ndim <= 1:
            no_decay_params.append(param)
        else:
            decay_params.append(param)

    param_groups = [
        {"params": decay_params,    "weight_decay": cfg.weight_decay},
        {"params": no_decay_params, "weight_decay": 0.0},
    ]

    optimizer = torch.optim.AdamW(
        param_groups,
        lr=cfg.learning_rate,
        betas=(cfg.beta1, cfg.beta2),
        eps=cfg.eps,
    )
    return optimizer


# ---------------------------------------------------------------------------
# 3. LR Scheduler  (linear warmup + cosine decay to min_lr)
# ---------------------------------------------------------------------------

def get_lr(step: int, cfg: TrainConfig) -> float:
    """
    Return the learning rate for a given optimiser step.

    Schedule:
      [0, warmup_steps)          → linear 0 → lr
      [warmup_steps, max_steps]  → cosine decay lr → min_lr
      > max_steps                → min_lr (clamped)
    """
    if step < cfg.warmup_steps:
        return cfg.learning_rate * (step / max(1, cfg.warmup_steps))

    if step >= cfg.max_steps:
        return cfg.min_lr

    # Cosine decay phase
    progress = (step - cfg.warmup_steps) / max(1, cfg.max_steps - cfg.warmup_steps)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return cfg.min_lr + cosine * (cfg.learning_rate - cfg.min_lr)


def build_scheduler(optimizer: torch.optim.AdamW, cfg: TrainConfig):
    """
    Return a LambdaLR scheduler wrapping the get_lr function.

    scheduler.step() must be called AFTER optimizer.step().
    The lambda receives the current step count (maintained internally by PyTorch).
    """
    # LambdaLR multiplies base_lr by the returned factor.
    # Our get_lr already returns the absolute LR, so we divide by base lr.
    def lr_lambda(step: int) -> float:
        return get_lr(step, cfg) / cfg.learning_rate

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
    return scheduler


# ---------------------------------------------------------------------------
# 4. Loss / metrics
# ---------------------------------------------------------------------------

def compute_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """
    Cross-entropy loss for language modelling.

    logits:  [B, T, vocab_size]
    targets: [B, T]              (int64)

    Returns scalar loss.
    """
    B, T, V = logits.shape
    loss = nn.functional.cross_entropy(
        logits.view(B * T, V),
        targets.view(B * T).long(),
    )
    return loss


def perplexity(loss: float) -> float:
    """
    Perplexity = exp(loss).  Clamps to avoid inf for very large losses.
    """
    try:
        return math.exp(min(loss, 20.0))   # exp(20) ≈ 485M — sensible cap
    except OverflowError:
        return float("inf")


# ---------------------------------------------------------------------------
# 5. AMP helper
# ---------------------------------------------------------------------------

def _amp_available(cfg: TrainConfig, device: torch.device) -> bool:
    """True only when AMP is requested AND a CUDA device is active."""
    return cfg.use_amp and device.type == "cuda"


# ---------------------------------------------------------------------------
# 6. Training step
# ---------------------------------------------------------------------------

def training_step(
    model: nn.Module,
    batch_iter,
    optimizer: torch.optim.AdamW,
    scheduler,
    scaler: "torch.cuda.amp.GradScaler",
    cfg: TrainConfig,
    device: torch.device,
    use_amp: bool,
) -> dict:
    """
    Execute one *optimiser* step, which may span cfg.grad_accum_steps micro-batches.

    Returns a dict with:
        loss        – mean loss over accum steps
        grad_norm   – gradient global norm (before clipping), or None
        tokens      – number of tokens processed in this optimiser step
        lr          – learning rate after this step
    """
    model.train()
    optimizer.zero_grad(set_to_none=True)

    accum_loss = 0.0
    total_tokens = 0

    for micro_step in range(cfg.grad_accum_steps):
        x, y = next(batch_iter)
        x = x.to(device)
        y = y.to(device)
        total_tokens += x.numel()

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(x)
            # Scale loss by grad_accum_steps so gradients average over all microbatches
            loss = compute_loss(logits, y) / cfg.grad_accum_steps

        if use_amp:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        accum_loss += loss.item()

    # Gradient clipping
    grad_norm = None
    if cfg.grad_clip > 0.0:
        if use_amp:
            scaler.unscale_(optimizer)
        grad_norm = nn.utils.clip_grad_norm_(
            model.parameters(), cfg.grad_clip
        ).item()

    # Optimiser step
    if use_amp:
        scaler.step(optimizer)
        scaler.update()
    else:
        optimizer.step()

    # Scheduler step (AFTER optimiser step)
    scheduler.step()

    return {
        "loss": accum_loss,          # already averaged by /grad_accum_steps
        "grad_norm": grad_norm,
        "tokens": total_tokens,
        "lr": scheduler.get_last_lr()[0],
    }


# ---------------------------------------------------------------------------
# 7. Evaluation
# ---------------------------------------------------------------------------

@torch.no_grad()
def evaluate(
    model: nn.Module,
    dataset: Dataset,
    cfg: TrainConfig,
    device: torch.device,
    use_amp: bool,
) -> dict:
    """
    Evaluate model on up to cfg.eval_batches random batches from dataset.

    Saves and restores model training/eval state.

    Returns dict with:
        val_loss  – mean cross-entropy
        val_ppl   – perplexity
    """
    was_training = model.training
    model.eval()

    loader = DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        drop_last=True,
    )

    total_loss = 0.0
    n_batches = 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(x)
            loss = compute_loss(logits, y)
        total_loss += loss.item()
        n_batches += 1
        if n_batches >= cfg.eval_batches:
            break

    model.train(was_training)

    mean_loss = total_loss / max(n_batches, 1)
    return {
        "val_loss": mean_loss,
        "val_ppl": perplexity(mean_loss),
    }


# ---------------------------------------------------------------------------
# 8. Random-state helpers
# ---------------------------------------------------------------------------

def get_rng_state() -> dict:
    state = {
        "python_rng": random.getstate(),
        "numpy_rng": np.random.get_state(),
        "torch_rng": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["cuda_rng"] = torch.cuda.get_rng_state_all()
    return state


def set_rng_state(state: dict) -> None:
    random.setstate(state["python_rng"])
    np.random.set_state(state["numpy_rng"])
    torch.set_rng_state(state["torch_rng"])
    if torch.cuda.is_available() and "cuda_rng" in state:
        torch.cuda.set_rng_state_all(state["cuda_rng"])


# ---------------------------------------------------------------------------
# 9. Checkpointing
# ---------------------------------------------------------------------------

CHECKPOINT_VERSION = 1


def save_checkpoint(
    output_dir: Path,
    step: int,
    model: nn.Module,
    optimizer: torch.optim.AdamW,
    scheduler,
    cfg: TrainConfig,
    model_cfg,
    best_val_loss: Optional[float] = None,
    rng_state: Optional[dict] = None,
) -> Path:
    """
    Atomically save a checkpoint.

    Writes to a temp file, then renames to avoid corruption on interruption.

    Saves:
    - model state_dict
    - optimizer state_dict
    - scheduler state_dict
    - step
    - TrainConfig (as dict)
    - LMConfig (as dict)
    - best_val_loss
    - rng_state
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    payload = {
        "version": CHECKPOINT_VERSION,
        "step": step,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "train_cfg": vars(cfg).copy(),
        "model_cfg": vars(model_cfg),
        "best_val_loss": best_val_loss,
        "rng_state": rng_state,
    }

    # Serialise Path → str so torch.save can handle it
    payload["train_cfg"]["output_dir"] = str(payload["train_cfg"]["output_dir"])

    step_name = f"step_{step:08d}.pt"
    final_path = output_dir / step_name

    # Write to temp file, then rename for atomicity
    fd, tmp_path = tempfile.mkstemp(dir=output_dir, suffix=".tmp")
    try:
        os.close(fd)
        torch.save(payload, tmp_path)
        shutil.move(tmp_path, final_path)
    except Exception:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
        raise

    # Update latest symlink / pointer file (plain text, always safe)
    latest_path = output_dir / "latest.txt"
    fd, tmp_latest = tempfile.mkstemp(dir=output_dir, suffix=".tmp")
    try:
        os.close(fd)
        with open(tmp_latest, "w") as f:
            f.write(step_name)
        shutil.move(tmp_latest, latest_path)
    except Exception:
        if os.path.exists(tmp_latest):
            os.remove(tmp_latest)
        raise

    return final_path


def load_checkpoint(
    checkpoint_path: Path,
    model: nn.Module,
    optimizer: Optional[torch.optim.AdamW] = None,
    scheduler=None,
    device: Optional[torch.device] = None,
    restore_rng: bool = True,
) -> dict:
    """
    Load a checkpoint and restore model (and optionally optimizer/scheduler) state.

    Returns the payload dict so callers can access step, config, etc.

    Raises ValueError for version mismatches.
    """
    if device is None:
        device = torch.device("cpu")

    payload = torch.load(checkpoint_path, map_location=device, weights_only=False)

    if payload.get("version") != CHECKPOINT_VERSION:
        raise ValueError(
            f"Checkpoint version mismatch. "
            f"Expected {CHECKPOINT_VERSION}, got {payload.get('version')}"
        )

    model.load_state_dict(payload["model_state"])

    if optimizer is not None:
        optimizer.load_state_dict(payload["optimizer_state"])

    if scheduler is not None:
        scheduler.load_state_dict(payload["scheduler_state"])

    if restore_rng and payload.get("rng_state") is not None:
        set_rng_state(payload["rng_state"])

    return payload


def get_latest_checkpoint(output_dir: Path) -> Optional[Path]:
    """Return the path pointed to by latest.txt, or None if not found."""
    latest_file = Path(output_dir) / "latest.txt"
    if not latest_file.exists():
        return None
    step_name = latest_file.read_text().strip()
    ckpt = Path(output_dir) / step_name
    return ckpt if ckpt.exists() else None


# ---------------------------------------------------------------------------
# 10. JSONL Logger
# ---------------------------------------------------------------------------

class JsonlLogger:
    """Append-only JSONL logger for training and evaluation metrics."""

    def __init__(self, log_path: Path):
        self.log_path = Path(log_path)
        self.log_path.parent.mkdir(parents=True, exist_ok=True)

    def log(self, record: dict) -> None:
        record = dict(record)
        record["timestamp"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
        with open(self.log_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


# ---------------------------------------------------------------------------
# 11. DataLoader factory
# ---------------------------------------------------------------------------

class _IterableWrapper:
    """Infinite iterator over a random-sampling DataLoader."""

    def __init__(self, dataset: Dataset, batch_size: int, num_workers: int):
        self._dataset = dataset
        self._batch_size = batch_size
        self._num_workers = num_workers
        self._loader = self._make_loader()
        self._iter = iter(self._loader)

    def _make_loader(self):
        return DataLoader(
            self._dataset,
            batch_size=self._batch_size,
            shuffle=True,
            num_workers=self._num_workers,
            drop_last=True,
            pin_memory=False,
        )

    def __next__(self):
        try:
            return next(self._iter)
        except StopIteration:
            self._iter = iter(self._make_loader())
            return next(self._iter)


# ---------------------------------------------------------------------------
# 12. Main training loop
# ---------------------------------------------------------------------------

def train(
    model: nn.Module,
    train_dataset: Dataset,
    val_dataset: Dataset,
    cfg: TrainConfig,
    model_cfg,
    resume_from: Optional[Path] = None,
    device: Optional[torch.device] = None,
) -> None:
    """
    Main training loop.

    Args:
        model:         DecoderOnlyLM (or any nn.Module with [B,T] → [B,T,V] forward)
        train_dataset: TokenDataset for training split
        val_dataset:   TokenDataset for validation split
        cfg:           TrainConfig
        model_cfg:     LMConfig (saved into checkpoints)
        resume_from:   Optional path to a checkpoint to resume from
        device:        torch.device; defaults to CUDA if available, else CPU
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = model.to(device)
    seed_everything(cfg.seed)

    use_amp = _amp_available(cfg, device)
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    optimizer = build_optimizer(model, cfg)
    scheduler = build_scheduler(optimizer, cfg)

    start_step = 0
    best_val_loss = float("inf")

    # Resume from checkpoint
    if resume_from is not None:
        payload = load_checkpoint(
            resume_from, model, optimizer, scheduler, device, restore_rng=True
        )
        start_step = payload["step"]
        best_val_loss = payload.get("best_val_loss") or float("inf")
        print(f"Resumed from step {start_step}")

    # Advance scheduler to match resumed step without modifying parameters
    # (LambdaLR state is already restored from checkpoint scheduler_state_dict)

    logger = JsonlLogger(cfg.output_dir / "train_log.jsonl")
    batch_iter = _IterableWrapper(train_dataset, cfg.batch_size, cfg.num_workers)

    total_tokens = 0
    t_start = time.time()

    print(
        f"Training {cfg.language} | device={device} | amp={use_amp} | "
        f"steps={cfg.max_steps} | batch={cfg.batch_size} | accum={cfg.grad_accum_steps}"
    )

    for step in range(start_step, cfg.max_steps):
        step_metrics = training_step(
            model, batch_iter, optimizer, scheduler, scaler, cfg, device, use_amp
        )
        total_tokens += step_metrics["tokens"]
        current_step = step + 1

        # ---- Logging ----
        if current_step % cfg.log_interval == 0:
            elapsed = time.time() - t_start
            record = {
                "type": "train",
                "step": current_step,
                "loss": round(step_metrics["loss"], 6),
                "lr": step_metrics["lr"],
                "grad_norm": step_metrics["grad_norm"],
                "tokens": total_tokens,
                "elapsed_s": round(elapsed, 2),
            }
            logger.log(record)
            print(
                f"step {current_step:6d} | loss {step_metrics['loss']:.4f} "
                f"| lr {step_metrics['lr']:.2e} | tokens {total_tokens:,}"
            )

        # ---- Evaluation ----
        if current_step % cfg.eval_interval == 0:
            eval_metrics = evaluate(model, val_dataset, cfg, device, use_amp)
            record = {
                "type": "eval",
                "step": current_step,
                "val_loss": round(eval_metrics["val_loss"], 6),
                "val_ppl": round(eval_metrics["val_ppl"], 4),
            }
            logger.log(record)
            print(
                f"  [eval] step {current_step} | val_loss {eval_metrics['val_loss']:.4f} "
                f"| ppl {eval_metrics['val_ppl']:.2f}"
            )
            if eval_metrics["val_loss"] < best_val_loss:
                best_val_loss = eval_metrics["val_loss"]

        # ---- Checkpointing ----
        if current_step % cfg.checkpoint_interval == 0:
            rng_state = get_rng_state()
            ckpt_path = save_checkpoint(
                cfg.output_dir, current_step, model, optimizer, scheduler,
                cfg, model_cfg, best_val_loss, rng_state
            )
            print(f"  [ckpt] saved {ckpt_path}")

    print(f"Training complete. Total tokens: {total_tokens:,}")


## 3. Device and Path Setup


In [ ]:

# Path configuration
KAGGLE_DATASET_ROOT = Path("/kaggle/input/datasets/negative00/phase2-cloud-data")
HINDI_TRAIN_DIR = KAGGLE_DATASET_ROOT / "hindi/data/splits/train"
HINDI_VAL_DIR = KAGGLE_DATASET_ROOT / "hindi/data/splits/val"
HINDI_TOKENIZER = KAGGLE_DATASET_ROOT / "hindi/tokenizer/models/hindi_bpe_10k_final.model"

OUTPUT_DIR = Path("/kaggle/working/phase2_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BIN_DIR = OUTPUT_DIR / "data_bin" / "hindi"
BIN_DIR.mkdir(parents=True, exist_ok=True)

HINDI_TRAIN_BIN = BIN_DIR / "train.bin"
HINDI_VAL_BIN = BIN_DIR / "val.bin"

# Device verification
print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU Available: True")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    
    # Calculate available VRAM
    t = torch.cuda.get_device_properties(0).total_memory
    print(f"Total VRAM: {t / 1024**3:.2f} GB")
else:
    print("GPU Available: False")
    sys.exit("CUDA GPU is required for full training. Enable a Kaggle GPU runtime before running this notebook.")


## 4. Hindi Data Conversion


In [ ]:

# Verification before conversion
def verify_paths():
    assert HINDI_TRAIN_DIR.exists(), f"Train dir missing: {HINDI_TRAIN_DIR}"
    assert HINDI_VAL_DIR.exists(), f"Val dir missing: {HINDI_VAL_DIR}"
    assert HINDI_TOKENIZER.exists(), f"Tokenizer missing: {HINDI_TOKENIZER}"
    assert OUTPUT_DIR.exists(), "Output dir missing"

verify_paths()

def conditional_convert(input_dir, output_bin, split_name):
    if output_bin.exists() and output_bin.stat().st_size > 0:
        print(f"{split_name}.bin already exists and is valid. Skipping conversion.")
        return None
    
    print(f"Converting {split_name}...")
    stats = convert_split(
        input_dir=input_dir,
        output_path=output_bin,
        tokenizer_path=HINDI_TOKENIZER,
        text_field="text"
    )
    
    # Simple post-conversion verification
    assert output_bin.exists(), f"Failed to create {output_bin}"
    sz = output_bin.stat().st_size
    assert sz > 0, f"{output_bin} is empty"
    assert sz == stats["token_count"] * 2, f"Size {sz} mismatch for token_count {stats['token_count']}"
    assert stats["token_min"] >= 0, f"Token min out of bounds: {stats['token_min']}"
    assert stats["token_max"] < 10000, f"Token max out of bounds: {stats['token_max']}"
    print(f"{split_name} conversion complete. File size: {sz / 1024**2:.2f} MB")
    return stats

train_stats = conditional_convert(HINDI_TRAIN_DIR, HINDI_TRAIN_BIN, "train")
val_stats = conditional_convert(HINDI_VAL_DIR, HINDI_VAL_BIN, "val")

if train_stats or val_stats:
    meta_dict = {}
    if train_stats: meta_dict["train"] = train_stats
    if val_stats: meta_dict["val"] = val_stats
    save_metadata(BIN_DIR, "hindi", meta_dict, HINDI_TOKENIZER.name)


## 5. Model Creation


In [ ]:

lm_cfg = LMConfig(max_seq_len=CONTEXT_LENGTH)
model = DecoderOnlyLM(lm_cfg)

parameter_count = sum(p.numel() for p in model.parameters())
print(f"Model Parameter Count: {parameter_count:,}")
assert parameter_count == 25_871_504, f"Parameter count mismatch: {parameter_count}"


## 6. Training


In [ ]:

train_cfg = TrainConfig(
    language="hindi",
    run_name="run_hindi",
    context_length=CONTEXT_LENGTH,
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    min_lr=MIN_LEARNING_RATE,
    max_steps=MAX_STEPS,
    warmup_steps=WARMUP_STEPS,
    eval_interval=EVAL_INTERVAL,
    checkpoint_interval=CHECKPOINT_INTERVAL,
    log_interval=LOG_INTERVAL,
    weight_decay=WEIGHT_DECAY,
    grad_clip=MAX_GRAD_NORM,
    seed=RANDOM_SEED,
    output_dir=OUTPUT_DIR / "hindi_checkpoints"
)

# Datasets
train_ds = TokenDataset(HINDI_TRAIN_BIN, context_length=CONTEXT_LENGTH)
val_ds = TokenDataset(HINDI_VAL_BIN, context_length=CONTEXT_LENGTH)

train_tokens = len(train_ds)
val_tokens = len(val_ds)
print(f"Train positions: {train_tokens:,}")
print(f"Val positions: {val_tokens:,}")

resume_ckpt = get_latest_checkpoint(train_cfg.output_dir)
if resume_ckpt:
    print(f"Resuming training automatically from {resume_ckpt}")

# Execution
train(
    model=model,
    train_dataset=train_ds,
    val_dataset=val_ds,
    cfg=train_cfg,
    model_cfg=lm_cfg,
    resume_from=resume_ckpt
)


## 7. Final Outputs


In [ ]:

import json
import matplotlib.pyplot as plt

log_file = train_cfg.output_dir / "train_log.jsonl"
steps = []
train_loss = []
val_steps = []
val_loss = []
best_val = None
final_val = None
final_ppl = None
actual_steps = 0

if log_file.exists():
    with open(log_file, "r", encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            if rec["type"] == "train":
                steps.append(rec["step"])
                train_loss.append(rec["loss"])
                actual_steps = max(actual_steps, rec["step"])
            elif rec["type"] == "eval":
                val_steps.append(rec["step"])
                vl = rec["val_loss"]
                val_loss.append(vl)
                if best_val is None or vl < best_val:
                    best_val = vl
                final_val = vl
                final_ppl = rec["val_ppl"]

    # Plot
    plt.figure(figsize=(10, 6))
    plt.plot(steps, train_loss, label="Train Loss", alpha=0.7)
    if val_steps:
        plt.plot(val_steps, val_loss, label="Val Loss", marker="o", color="red")
    plt.xlabel("Steps")
    plt.ylabel("Cross-Entropy Loss")
    plt.title("Hindi Language Model Training Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plot_path = OUTPUT_DIR / "loss_plot.png"
    plt.savefig(plot_path)
    print(f"Loss plot saved to {plot_path}")
    plt.show()

# Final Summary Print
print("\n" + "="*50)
print("FINAL COMPLETION SUMMARY")
print("="*50)
print(f"Device:                 {'CUDA' if torch.cuda.is_available() else 'CPU'}")
print(f"GPU Name:               {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"Parameter Count:        {sum(p.numel() for p in model.parameters()):,}")
print(f"Hindi Train Tokens:     {train_tokens:,} (approx.)")
print(f"Hindi Val Tokens:       {val_tokens:,} (approx.)")
print(f"Configured Max Steps:   {MAX_STEPS:,}")
print(f"Actual Completed Steps: {actual_steps:,}")
print(f"Best Validation Loss:   {best_val if best_val else 'N/A'}")
print(f"Final Validation Loss:  {final_val if final_val else 'N/A'}")
print(f"Final Val Perplexity:   {final_ppl if final_ppl else 'N/A'}")
print(f"Checkpoint Directory:   {train_cfg.output_dir}")
print(f"Output Directory:       {OUTPUT_DIR}")
print("="*50)
